In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import bokeh.palettes
import scipy.stats.mstats
import numpy as np
import json

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Palatino"],
})

In [2]:
with open('memory-usage.json', 'r') as f:
    benchmarks = json.load(f)

In [3]:
# get the names of the vms
vms = [vm['vm'] for vm in benchmarks[0]["vms"]]
# create an array of numpy arrays of times; rows are vm, columns are benchmarks
times = np.array([[b["vms"][vm]["time"] for b in benchmarks] for vm in range(len(vms))])
# weights are the times of the first vm (local TrueJIT)
weights = np.array(times[0])
# calculate speedups compared to the first vm
speedups = []
for i in range(1, len(vms)):
    speedups.append(times[0] / times[i])
speedups.insert(0, np.ones(len(benchmarks)))
speedups = np.array(speedups)
# calculate the geometric mean of speedups
gmeans = [scipy.stats.mstats.gmean(speedup[speedup != 0], weights=weights[speedup != 0]) for speedup in speedups]

# sort by gmeans
vms_gmeans = list(zip(vms, gmeans))
vms_gmeans.sort(key=lambda x: x[1])
vms, gmeans = zip(*vms_gmeans)

# plot
fig, ax = plt.subplots(figsize=(3, 2), dpi=320)
colors = bokeh.palettes.inferno(len(vms))

ax.bar(vms, gmeans, width=.5, color=colors, edgecolor='black', linewidth=0.5)

# write the speedup on top of the bars
for i, gmean in enumerate(gmeans):
    ax.text(i, gmean + 0.1, f"{gmean:.2f}", ha='center', va='center', fontsize=5, color='black')

# x-axis are 45 degrees rotated
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=8)

# make the x label for 'truejit-local' red
for i, vm in enumerate(vms):
    if vm == 'truejit-warm':
        ax.get_xticklabels()[i].set_color('red')

# y-limit
ax.set_ylim(0, 1.5)
# ticks
ax.tick_params(axis='both', which='major', labelsize=6)

ax.set_xlabel('Virtual Machines', fontsize=6)
ax.set_ylabel('Normalised Speedup\n(truejit-local = 1)', fontsize=6)

fig.tight_layout()

# to pdf
plt.savefig('out/speedup.pdf', format='pdf', bbox_inches='tight', dpi=320)

plt.show()

In [4]:
# get the names of the vms
vms = [vm['vm'] for vm in benchmarks[0]["vms"]]
# create an array of numpy arrays of memories; rows are vm, columns are benchmarks
memories = np.array([[b["vms"][vm]["memory"] for b in benchmarks] for vm in range(len(vms))])
# calculate memory usage compared to the first vm
memory_usages = []
for i in range(1, len(vms)):
    memory_usages.append(memories[i] / memories[0])
memory_usages.insert(0, np.ones(len(benchmarks)))
memory_usages = np.array(memory_usages)
# calculate the geometric mean of memory usages
gmeans = [scipy.stats.mstats.gmean(memory_usage[memory_usage != np.inf], weights=weights[memory_usage != np.inf]) for
          memory_usage in memory_usages]

# sort by gmeans
vms_gmeans = list(zip(vms, gmeans))
vms_gmeans.sort(key=lambda x: x[1], reverse=True)
vms, gmeans = zip(*vms_gmeans)

# plot
fig, ax = plt.subplots(figsize=(3, 2), dpi=320)
colors = bokeh.palettes.inferno(len(vms))

ax.bar(vms, gmeans, width=.5, color=colors, edgecolor='black', linewidth=0.5)

# write the speedup on top of the bars
for i, gmean in enumerate(gmeans):
    ax.text(i, gmean + 0.1, f"{gmean:.2f}", ha='center', va='center', fontsize=5, color='black')

# x-axis are 45 degrees rotated
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=8)

# make the x label for 'truejit-local' red
for i, vm in enumerate(vms):
    if vm == 'truejit-warm':
        ax.get_xticklabels()[i].set_color('red')

# y-limit
ax.set_ylim(0, 2)
# ticks
ax.tick_params(axis='both', which='major', labelsize=6)

ax.set_xlabel('Virtual Machines', fontsize=6)
ax.set_ylabel('Normalised Memory Usage\n(truejit-local = 1)', fontsize=6)

fig.tight_layout()

# to pdf
plt.savefig('out/memory_usage.pdf', format='pdf', bbox_inches='tight', dpi=320)

plt.show()

In [5]:
import numpy as np

# get the memory consumption of the truejit local
local_memories = np.asarray([b["vms"][0]["memory"] / 1000 for b in benchmarks])
local_memories
# get the memory consumption of the truejit remote
remote_memories = np.asarray([b["vms"][1]["memory"] / 1000 for b in benchmarks])
remote_memories

# make remote memory usage relative to local memory usage
remote_memories /= local_memories

# calculate the geometric mean of remote memory usage
gmean = scipy.stats.mstats.gmean(remote_memories)
# add the gmean to the local_memories
remote_memories = np.append(remote_memories, gmean)



# show next to each other
fig, ax = plt.subplots(figsize=(len(benchmarks) / 4, 1), dpi=320)

# style
width = .5
colors = bokeh.palettes.inferno(2)
ax.margins(0.01)

# only plot the remote memory usage
ax.bar(np.arange(len(remote_memories)) + (width / 2), remote_memories, width, color='red', edgecolor='black', linewidth=0.5)

# x-axis (always put it after bars)
ax.set_xticks(np.arange(len(remote_memories)) + width / 2)
ax.set_xticklabels([b["benchmark"] for b in benchmarks + [{"benchmark": "GeoMean"}]])
ax.set_xlabel('Benchmarks')

# rotate the x-axis labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# y-axis lim is 0 to 1
ax.set_ylim(0, 1)

plt.show()


In [6]:
# calculate the weighted geometric mean of local and remote memory usage
local_memory = sum(local_memories) / len(local_memories)
remote_memory = sum(remote_memories) / len(remote_memories)

# calculate memory usage improvement of remote compared to local
improvement = (remote_memory - local_memory) / remote_memory
print(f"Local Memory: {local_memory:.2f}MB")
print(f"Remote Memory: {remote_memory:.2f}MB")
print(f"Improvement: {improvement:.2f}")

In [7]:
# Local Memory: 302.76MB
# Remote Memory: 264.29MB
# Improvement: 1.15

In [8]:
# get the names of the vms
vms = [vm['vm'] for vm in benchmarks[0]["vms"]]
# create an array of numpy arrays of times; rows are vm, columns are benchmarks
times = np.array([[b["vms"][vm]["time"] for b in benchmarks] for vm in range(len(vms))])
# calculate the speedup of each vm compared to the first vm
speedups = []
for i in range(1, len(vms)):
    speedups.append(times[0] / times[i])

# plot times, for each benchmark plot all vm times next to each other
fig, ax = plt.subplots(figsize=(len(benchmarks) / 2, 4), dpi=320)

colors = bokeh.palettes.inferno(len(vms))
for i, vm in enumerate(vms):
    ax.bar(np.arange(len(benchmarks)) + i * .6 / len(vms), times[i], width=.6 / len(vms), color=colors[i],
           edgecolor='black', linewidth=0.5)

# x-axis are 45 degrees rotated
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=8)

# x labels
ax.set_xticks(np.arange(len(benchmarks)) + .3)
ax.set_xticklabels([b["benchmark"] for b in benchmarks], fontsize=8)

# y is log
ax.set_yscale('log')

# margin is 0
ax.margins(0)


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import bokeh.palettes

# get the names of the vms
vms = [vm['vm'] for vm in benchmarks[0]["vms"]]
# create an array of numpy arrays of times; rows are vm, columns are benchmarks
times = np.array([[b["vms"][vm]["time"] for b in benchmarks] for vm in range(len(vms))])
# weights are the times of the first vm (local TrueJIT)
weights = np.array(times[0])
# calculate speedups compared to the first vm
speedups = []
for i in range(1, len(vms)):
    speedups.append(times[0] / times[i])
speedups.insert(0, np.ones(len(benchmarks)))
speedups = np.array(speedups)
# calculate the geometric mean of speedups
gmeans = [gmean(speedup[speedup != 0], weights=weights[speedup != 0]) for speedup in speedups]

fig, ax = plt.subplots(figsize=(len(benchmarks) / 3, 4), dpi=320)

# style
width = .6 / len(vms)
colors = bokeh.palettes.inferno(len(vms) + 2)[1:-1]
ax.margins(0.01)

times.head()

# plot times
xs = np.arange(len(benchmarks))
for i, vm in enumerate(vms):
    ax.bar(xs + i * width, times[i], width, label=vm, color=colors[i], edgecolor='black', linewidth=0.5)

# x-axis (always put it after bars)
ax.set_xticks(xs)
ax.set_xticklabels([b["benchmark"] for b in benchmarks])
ax.set_xlabel('Benchmarks')

# rotate the x-axis labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# y-axis log
ax.set_yscale('log')
# labels are 0.001, 0.01, 0.1, 1, 10
ax.set_yticks([0.01, 0.01, 0.1, 1, 10])
ax.set_yticklabels([f"{i:.2f}" for i in [0.01, 0.01, 0.1, 1, 10]])

# draw a horizontal dashed line at y=1
ax.axhline(1, color='black', linestyle='--', linewidth=0.5)

ax.set_ylabel('Normalized Execution Time\n(Local TrueJIT Cold = 1.00)')

ax.legend(loc='upper center',
          ncol=len(vms),
          frameon=False,
          bbox_to_anchor=(0.5, 1.17),
          fontsize=8,
          bbox_transform=ax.transAxes)

fig.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bokeh.palettes

fig, ax = plt.subplots(figsize=(len(benchmarks) / 3, 4), dpi=320)

# style
number_of_vms = len(vms)
width = .6 / number_of_vms
colors = bokeh.palettes.inferno(number_of_vms + 2)[1:-1]
ax.margins(0.01)

# add weighted geometric mean of time for each vms as the last benchmark use scipy.stats.mstats.gmean
benchmarks.append({"benchmark": "weighted geomean", "vms": []})
for vm in range(number_of_vms):
    times = [b["vms"][vm]["time"] for b in benchmarks[:-1]]
    times = [t for t in times if t != 0]
    benchmarks[-1]["vms"].append({"vm": vms[vm], "time": gmean(times, weights=times)})
print(benchmarks[-1])

# bars
xs = np.arange(len(benchmarks))
vms = [[b["vms"][i] for b in benchmarks] for i in range(number_of_vms)]

# make times of all vms relative to the first vm
for i in range(number_of_vms):
    if i == 1:
        continue
    for j in range(len(benchmarks)):
        vms[i][j]["time"] /= vms[1][j]["time"]
# make the first vm 1
baseline = []
for j in range(len(benchmarks)):
    baseline.append(vms[1][j]["time"])
    vms[1][j]["time"] = 1

base_x = number_of_vms // 2
for i, vm in enumerate(vms):
    times = [v["time"] for v in vm]
    ax.bar(xs + (i - base_x) * width, times, width, label=vm[0]["vm"], color=colors[i], edgecolor='black',
           linewidth=0.5)
    # write the baseline on top of the first vm rotate the text 90 degrees
    if i == 1:
        for j, time in enumerate(times):
            ax.text(j - i * width, 15, f"{baseline[j]:.1f}s", ha='center', va='center', rotation=45, fontsize=10)

# x-axis (always put it after bars)
ax.set_xticks(xs)
ax.set_xticklabels([b["benchmark"] for b in benchmarks])
ax.set_xlabel('Benchmarks')

# rotate the x-axis labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# y-axis log
ax.set_yscale('log')
# labels are 0.001, 0.01, 0.1, 1, 10
ax.set_yticks([0.01, 0.01, 0.1, 1, 10])
ax.set_yticklabels([f"{i:.2f}" for i in [0.01, 0.01, 0.1, 1, 10]])

# draw a horizontal dashed line at y=1
ax.axhline(1, color='black', linestyle='--', linewidth=0.5)

ax.set_ylabel('Normalized Execution Time\n(Local TrueJIT Cold = 1.00)')

ax.legend(loc='upper center',
          ncol=6,
          frameon=False,
          bbox_to_anchor=(0.5, 1.17),
          fontsize=8,
          bbox_transform=ax.transAxes)

fig.tight_layout()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(len(benchmarks) / 3, 4), dpi=320)

# style
number_of_vms = 5
width = .6 / number_of_vms
colors = bokeh.palettes.inferno(number_of_vms + 2)[1:-1]
# labels_fontsize = 12
ax.margins(0.01)

# add weighted geometric mean of memory for each vms as the last benchmark use scipy.stats.mstats.gmean
benchmarks.append({"benchmark": "weighted geomean", "vms": []})
for i in range(number_of_vms):
    memory = [b["vms"][i]["memory"] for b in benchmarks[:-1]]
    # remove 0s from the list
    memory = [m for m in memory if m != 0]
    benchmarks[-1]["vms"].append({"vm": benchmarks[0]["vms"][i]["vm"], "memory": gmean(memory, weights=memory)})

# bars
xs = np.arange(len(benchmarks))
vms = [[b["vms"][i] for b in benchmarks] for i in range(number_of_vms)]

# make memories of all vms relative to the first vm
for i in range(number_of_vms):
    if i == 1:
        continue
    for j in range(len(benchmarks)):
        vms[i][j]["memory"] /= vms[1][j]["memory"]

# make the first vm 1
baseline = []
for j in range(len(benchmarks)):
    baseline.append(vms[1][j]["memory"] // 1000)
    vms[1][j]["memory"] = 1

base_x = number_of_vms // 2
for i, vm in enumerate(vms):
    memories = [v["memory"] for v in vm]
    ax.bar(xs + (i - base_x) * width, memories, width, label=vm[0]["vm"], color=colors[i], edgecolor='black',
           linewidth=0.5)
    # write the baseline on top of the first vm rotate the text 90 degrees
    if i == 1:
        for j, memory in enumerate(memories):
            ax.text(j - i * width, 5, f"{baseline[j]}MB", ha='center', va='center', rotation=45, fontsize=8)

# x-axis (always put it after bars)
ax.set_xticks(xs)
ax.set_xticklabels([b["benchmark"] for b in benchmarks])
ax.set_xlabel('Benchmarks')

# rotate the x-axis labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# y-axis log
ax.set_yscale('log')
# set limits
ax.set_ylim(0.01, 10)
# labels are 0.001, 0.01, 0.1, 1, 10
ax.set_yticks([0.01, 0.01, 0.1, 1, 10])
ax.set_yticklabels([f"{i:.2f}" for i in [0.01, 0.01, 0.1, 1, 10]])

# draw a horizontal dashed line at y=1
ax.axhline(1, color='black', linestyle='--', linewidth=0.5)

ax.set_ylabel('Normalized Memory Usage\n(Local True JIT Cold = 1.00')

ax.legend(loc='upper center',
          ncol=6,
          frameon=False,
          bbox_to_anchor=(0.5, 1.17),
          fontsize=8,
          bbox_transform=ax.transAxes)

fig.tight_layout()

plt.show()